In [0]:

# Databricks notebook source
# POC-2: Retail Order CDC & Revenue Lakehouse
# Phase 0/1: Environment and Source Validation
# READ-ONLY: Does not modify source files or Delta tables.

from pyspark.sql import functions as F
from pyspark.sql.types import StructType
from collections import Counter
import json

CATALOG = "retail_poc2"
VOLUME_PATH = "/Volumes/retail_poc2/landing/source_files"

EXPECTED_SCHEMAS = ["landing", "bronze", "silver", "gold", "audit"]
EXPECTED_DIRECTORIES = ["initial", "CDC", "test_cases"]
EXPECTED_INITIAL_FILES = [
    "customers.csv",
    "products.csv",
    "orders.csv"
]

failures = []
warnings = []

def check(name, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"[{status}] {name} {detail}")
    if not condition:
        failures.append(name)

def inspect_csv(path):
    print(f"\n--- Inspecting {path} ---")
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("mode", "FAILFAST")
        .csv(path)
    )

    print("Columns:", df.columns)
    print("Inferred schema:")
    df.printSchema()

    count = df.count()
    print("Row count:", count)

    check(f"Nonempty file: {path}", count > 0)
    check(
        f"No duplicate column names: {path}",
        len(df.columns) == len(set(df.columns))
    )

    return df, count

print("=" * 65)
print("POC-2 ENVIRONMENT AND SOURCE VALIDATION")
print("=" * 65)

# 1. Spark environment
print("\n1. SPARK ENVIRONMENT")
print("Spark version:", spark.version)

check("Spark session available", spark is not None)

try:
    delta_sql = spark.sql(
        "SELECT current_catalog() AS catalog, current_schema() AS schema"
    )
    delta_sql.show(truncate=False)
    check("Spark SQL available", True)
except Exception as e:
    check("Spark SQL available", False, str(e))

# 2. Catalog and schema access
print("\n2. UNITY CATALOG")

try:
    catalogs = [
        r.catalog for r in spark.sql("SHOW CATALOGS").collect()
    ]
    check(f"Catalog {CATALOG} exists", CATALOG in catalogs)

    schemas = [
        r.databaseName
        for r in spark.sql(
            f"SHOW SCHEMAS IN `{CATALOG}`"
        ).collect()
    ]

    for schema in EXPECTED_SCHEMAS:
        check(
            f"Schema {CATALOG}.{schema}",
            schema in schemas
        )

except Exception as e:
    check("Unity Catalog access", False, str(e))

# 3. Delta Lake availability
print("\n3. DELTA LAKE")

try:
    from delta.tables import DeltaTable
    check("Delta Lake Python API available", True)
except Exception as e:
    check("Delta Lake Python API available", False, str(e))

# 4. Volume and source directories
print("\n4. SOURCE VOLUME")

try:
    root_items = dbutils.fs.ls(VOLUME_PATH)
    root_names = [
        item.name.rstrip("/") for item in root_items
    ]
    check("Source Volume accessible", True)

    for directory in EXPECTED_DIRECTORIES:
        check(
            f"Directory {directory}/ exists",
            directory in root_names
        )
except Exception as e:
    check("Source Volume accessible", False, str(e))

# 5. Initial source files
print("\n5. INITIAL SOURCE DATA")

initial_dataframes = {}

for filename in EXPECTED_INITIAL_FILES:
    path = f"{VOLUME_PATH}/initial/{filename}"
    try:
        df, count = inspect_csv(path)
        initial_dataframes[filename] = df
    except Exception as e:
        check(f"Read {filename}", False, str(e))

# 6. CDC files
print("\n6. CDC DATA")


# 6. CDC files
print("\n6. CDC DATA")
cdc_files = []

try:
    cdc_files = sorted([
        item.path
        for item in dbutils.fs.ls(f"{VOLUME_PATH}/CDC")
        if item.name.lower().endswith(".csv")
    ])

    check("CDC files exist", len(cdc_files) > 0)
    print("CDC files:", cdc_files)

    for path in cdc_files:
        df, count = inspect_csv(path)

        required_cdc_columns = [
            "event_id",
            "order_id",
            "operation",
            "event_sequence",
            "event_timestamp"
        ]

        missing = [
            column for column in required_cdc_columns
            if column not in df.columns
        ]

        check(
            f"CDC required columns: {path}",
            len(missing) == 0,
            f"Missing: {missing}"
        )

        if "operation" in df.columns:
            operations = [
                row["operation"]
                for row in df.select("operation")
                .distinct()
                .collect()
            ]

            print("CDC operations:", operations)

            check(
                f"Valid CDC operations: {path}",
                set(operations).issubset({"I", "U", "D"})
            )

except Exception as e:
    check("CDC directory scan", False, str(e))


# 7. Test-case files
print("\n7. TEST DATA")

try:
    test_files = sorted([
        item.path
        for item in dbutils.fs.ls(
            f"{VOLUME_PATH}/test_cases"
        )
        if item.name.lower().endswith(".csv")
    ])

    check("Test-case files exist", len(test_files) > 0)

    for path in test_files:
        df, count = inspect_csv(path)

except Exception as e:
    check("Test-case directory scan", False, str(e))

# 8. Final report
print("\n" + "=" * 65)
print("VALIDATION SUMMARY")
print("=" * 65)

print("Failed checks:", len(failures))

for failure in failures:
    print("FAIL:", failure)

if warnings:
    print("Warnings:", warnings)

if failures:
    print("\nRESULT: VALIDATION FAILED")
    print("Review failures before building Bronze.")
else:
    print("\nRESULT: INITIAL VALIDATION PASSED")
    print("Next: approve source contracts and CDC/revenue rules.")

print("=" * 65)
